# Unit 10 — Observation history and action execution

## The idea before the implementation

**Diffusion is the action policy.** Diffusion Policy generates an action window directly from recent camera images and robot state. It starts with an action-shaped noise tensor, then repeatedly uses a learned denoiser and a scheduler to turn it into plausible actions. It does not refine ACT or VLA outputs in this course, and it does not generate camera images.

**Where you work.** You work on execution around the supplied model. Once a complete window has been sampled, your queue selects the portion aligned with the newest observation and sends one action per controller tick. It samples a new window only when the queue is empty. Lessons 11–13 open the training objective and denoiser; Lesson 14 opens the sampler.

**Pause and explain.** Keep three counts separate: past observations, predicted action slots, and internal denoising updates. Only an executed action advances the simulator.

Keep [the concept guide](../concepts/diffusion-policy.md) open beside this notebook for the full system diagrams and lesson map.

A Diffusion Policy uses recent observations to predict a future action window. The robot executes a short prefix, then predicts again when the queue is empty. This unit builds the queue operation used by the real policy callback.

In [1]:
import sys
import time
from collections import deque
from collections.abc import Callable, Mapping
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from torch import Tensor

WORKSPACE = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "manifests/act_references.yaml").is_file()
)
sys.path.insert(0, str(WORKSPACE / "src"))
from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.diffusion import (
    CallbackDiffusionPolicy,
    load_prepared_diffusion,
    make_step_noises,
)

## Three time ranges

<svg xmlns="http://www.w3.org/2000/svg" width="780" height="138" viewBox="0 0 780 138" role="img" aria-label="Align the action window to the newest observation">
  <title>Align the action window to the newest observation</title>
  <desc>Slot 0 aligns with the older observation. Slots 9 through 15 are unused in this example.</desc>
  <rect width="780" height="138" rx="8" fill="#f8fafc"/>
  <text x="12" y="22" font-family="sans-serif" font-size="15" font-weight="bold" fill="#0f172a">Align the action window to the newest observation</text>
  <rect x="12" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="97.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">History:</text>
  <text x="97.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">older and current</text>
  <path d="M 186 67 H 203 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="207" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="292.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Sample all</text>
  <text x="292.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">16 action slots</text>
  <path d="M 381 67 H 398 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="402" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="487.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Select slots</text>
  <text x="487.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">1 through 8</text>
  <path d="M 576 67 H 593 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="597" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="682.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Execute eight;</text>
  <text x="682.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">then predict again</text>
  <text x="12" y="120" font-family="sans-serif" font-size="12" fill="#334155">Slot 0 aligns with the older observation. Slots 9 through 15 are unused in this example.</text>
</svg>

With two observation steps, slot 1 aligns with the current observation. This is the first executed slot. The predicted horizon includes an older aligned slot; it is not entirely future actions.


`n_obs_steps` is the number of observations used as context. `horizon` is the number of predicted future slots. `n_action_steps` is the queue prefix sent to the controller. They are different counts.

**Timing example.** LIBERO control runs at 20 Hz, so one controller step is 0.05 s. Dataset timestamp labels use a 10 FPS convention (0.10 s label offsets). That storage convention is separate from the controller clock; it does not set the physical action duration. The policy sends one relative action for one controller step. Do not repeat an action because its dataset label spans 0.10 s.

For `n_obs_steps=2`, `horizon=16`, and `n_action_steps=8`, the newest observation aligns with action index 1. The queue therefore takes indices 1 through 8. `max_steps` is the total environment steps. `num_inference_steps` is the number of denoising updates for one prediction.

## Exercise 10.1 — Consume a predicted action queue

**Purpose.** Use each predicted window in order, and request a new window only when the queue is empty.

**Worked example.** If prediction values are `[0,1,2,3,4,5]`, the policy has two observation steps, and it executes three actions per prediction, the first call queues `[1,2,3]`. The next two calls consume `2` and `3`; they do not sample again. The fourth call needs a new prediction.

**Contract.** `predict_chunk` returns a tensor of shape $(B,H,A)$. The queue stores actions with shape $(B,A)$. When the queue is empty, validate the prefix and queue the predicted actions at these slots, in order:

$$j=n_o-1,\ n_o,\ \dots,\ n_o+n_a-2$$

Where:
* $B$ - batch size (1 for one environment).
* $H$ - `horizon`, the number of predicted slots.
* $A$ - action width (7 in LIBERO).
* $n_o$ - `n_obs_steps`; slot $n_o-1$ is aligned with the newest observation.
* $n_a$ - `n_action_steps`, the number of queued actions, so the queue receives exactly $n_a$ slots.
* $j$ - a slot index on the horizon axis of the prediction.

Pop and return one action. Leave the rest in the queue.

More behaviour checks for this exercise: `check_next_queued_action` in [lesson10_checks.py](lesson10_checks.py).

In [ ]:
def next_queued_action(
    queue: deque[Tensor], predict_chunk: Callable[[], Tensor], n_obs_steps: int, n_action_steps: int
) -> Tensor:
    """Predict only when empty, then return one (B,A) action."""
    if len(queue) == 0:
        chunk = predict_chunk()

        if chunk.shape[1] < n_action_steps + n_obs_steps - 1:
            raise ValueError("Pred horizon too short")

        for i in range(n_action_steps):
            queue.append(chunk[:, i + n_obs_steps - 1, :])
    return queue.popleft()


chunk = torch.arange(12, dtype=torch.float32).reshape(1, 6, 2)
queue = deque()
predictions = []


def predict() -> Tensor:
    predictions.append(1)
    return chunk


print(chunk.shape, chunk)
print(chunk[:, 1, :])
first = next_queued_action(queue, predict, 2, 3)
second = next_queued_action(queue, predict, 2, 3)
third = next_queued_action(queue, predict, 2, 3)
assert (
    first.tolist() == [[2.0, 3.0]]
    and second.tolist() == [[4.0, 5.0]]
    and third.tolist() == [[6.0, 7.0]]
), (first, second, third)
assert len(predictions) == 1 and len(queue) == 0, (
    "The policy predicts once, then consumes the queue."
)

# More behaviour checks: lesson10_checks.py, next to this notebook.
from lesson10_checks import check_next_queued_action

check_next_queued_action(next_queued_action)

torch.Size([1, 6, 2]) tensor([[[ 0.,  1.],
         [ 2.,  3.],
         [ 4.,  5.],
         [ 6.,  7.],
         [ 8.,  9.],
         [10., 11.]]])
tensor([[2., 3.]])
✓ queues slots n_o-1 .. n_o+n_a-2, pops them first-in-first-out, predicts only when empty
✓ n_obs_steps = 1 starts at slot 0; a prefix that exactly fills the horizon is allowed
✓ an existing queue is consumed from the front without predicting
✓ a prefix that does not fit the predicted horizon raises ValueError


## View the horizons

The sliders change only this drawing. They do not run or train a policy.

In [ ]:
def show_horizons(history: int = 2, horizon: int = 16, execute: int = 8) -> None:
    fig, ax = plt.subplots(figsize=(9, 2))
    ax.barh(2, history, left=-history, color="#4c78a8", label="observation history")
    ax.barh(1, horizon, left=0, color="#f2cf5b", label="predicted horizon")
    ax.barh(0, execute, left=history - 1, color="#54a24b", label="executed prefix")
    ax.set_yticks([0, 1, 2], ["queue", "prediction", "history"])
    ax.set_xlabel("controller time slots")
    ax.set_title(f"history={history}; horizon={horizon}; execute={execute}")
    ax.legend(ncols=3)
    plt.show()


display(
    widgets.interactive(
        show_horizons,
        history=widgets.IntSlider(value=2, min=1, max=4, description="history"),
        horizon=widgets.IntSlider(value=16, min=8, max=24, step=4, description="horizon"),
        execute=widgets.IntSlider(value=8, min=1, max=16, description="execute"),
    )
)

interactive(children=(IntSlider(value=2, description='history', max=4, min=1), IntSlider(value=16, description…

## Follow a rollout

At the first step, the policy repeats the first observation until its history is full. It predicts a horizon, queues an execution prefix, and removes one action per controller step. When the queue is empty, it predicts a new window from the latest history.

If a rollout has 20 total control steps and every queue prefix has 8 actions, the policy needs three predictions. The final prediction has four unused queued actions when the rollout stops. This is expected: `max_steps` ends the environment rollout.

**Compare.** Explain why the observation history can be shorter than the predicted horizon. State which of the three counts changes inference compute.

## Full experiment — Run the learner queue in LIBERO

The experiment uses the prepared image and state encoders, the policy sampler, this lesson's queue operation, and the standard observation/action processors. It runs one bounded 20-step rollout on a development state.

In [13]:
CHECKPOINT = WORKSPACE / ".local-data/runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
POLICY, PREPROCESS, POSTPROCESS = load_prepared_diffusion(
    CHECKPOINT, device="cuda" if torch.cuda.is_available() else "cpu"
)

Loading weights from local directory


In [14]:
class LearnerQueuePolicy(CallbackDiffusionPolicy):
    """Use the learner queue operation inside the standard policy callback."""

    @torch.no_grad()
    def select_action(self, batch: Mapping[str, Tensor]) -> Tensor:
        self._push(batch)
        config = self.policy.config

        def predict_chunk() -> Tensor:
            observations = {
                name: torch.stack(list(queue), dim=1) for name, queue in self._observations.items()
            }
            state = observations["observation.state"]
            action_feature = config.action_feature
            if action_feature is None:
                raise ValueError("Diffusion Policy requires an action feature")
            shape = (state.shape[0], config.horizon, action_feature.shape[0])
            generator = torch.Generator(device=state.device).manual_seed(
                self.seed + self.chunks_made
            )
            initial_noise = torch.randn(
                shape, generator=generator, device=state.device, dtype=state.dtype
            )
            step_noises = make_step_noises(
                self.num_inference_steps, shape, generator, state.device, state.dtype
            )
            started = time.perf_counter()
            window = self.predict_window(observations, initial_noise, step_noises)
            if state.device.type == "cuda":
                torch.cuda.synchronize()
            self.sample_seconds.append(time.perf_counter() - started)
            self.chunks_made += 1
            return window

        return next_queued_action(
            self._actions, predict_chunk, config.n_obs_steps, config.n_action_steps
        )


RUNNER = LearnerQueuePolicy(POLICY, num_inference_steps=5, seed=7)
ENV = open_libero_task("bowl_plate", 10, WORKSPACE / "configs", WORKSPACE / ".local-data")
try:
    RESULT = run_policy_episode(
        ENV,
        RUNNER,
        PREPROCESS,
        POSTPROCESS,
        "bowl_plate",
        10,
        WORKSPACE / "configs",
        max_steps=20,
        reset_seed=7,
    )
finally:
    ENV.close()
assert RESULT.steps <= 20 and RESULT.control_hz == 20
assert RUNNER.chunks_made >= 1
print({"steps": RESULT.steps, "success": RESULT.success, "sampler_calls": RUNNER.chunks_made})

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'steps': 20, 'success': False, 'sampler_calls': 3}
